# Grok Evaluation - With Reasoning Token Logging

This notebook evaluates Grok on the OI Benchmark and captures reasoning tokens.

**Install:** `pip install openai`

**Models:**
- `grok-3-mini` - Supports `reasoning_effort` (low/high)
- `grok-3` - No reasoning_effort parameter
- `grok-4` - Always max reasoning (no parameter needed)

In [8]:
from openai import OpenAI

api_key = "xai-YM20cgFAhB7VSuakMc3P9oXeiMQv2QJHBrEYwiY2yCuiL3t1NsrNb17GesiZbcYRcry5BfxdPGdsDZLe"  # ← Your xAI API key

client = OpenAI(
    api_key=api_key,
    base_url="https://api.x.ai/v1"
)

In [ ]:
import pandas as pd

df = pd.read_csv("OI_Benchmark.csv")  
print(f"Loaded {len(df)} questions")
df.head()

Loaded 1010 questions


,question_ID,compound.name_1,SMILES_1,compound.name_2,SMILES_2,OPTIONS,question_category,prompt.1,prompt.2,answer,other_info
0,5e8d4af0-4df1-4fcd-b7c7-b7fe3bf1b630,"2-phenylethyl acetate, ethyl 2-methylbutanoate...","CC(=O)OCCC1=CC=CC=C1, CCC(C)C(=O)OCC, CCCCC1CC...",NaN,NaN,apple;mango;chocolate;peanut,smell_identification,"Given the molecules CC(=O)OCCC1=CC=CC=C1, CCC(...","Given the molecules 2-phenylethyl acetate, eth...",mango,NaN
1,b9df1df5-bbe7-4069-8a5f-976c699d87ad,"benzyl alcohol, benzaldehyde, phenylacetic aci...","C1=CC=C(C=C1)CO, C1=CC=C(C=C1)C=O, C1=CC=C(C=C...",NaN,NaN,peanut;fish;apple;walnut,smell_identification,"Given the molecules C1=CC=C(C=C1)CO, C1=CC=C(C...","Given the molecules benzyl alcohol, benzaldehy...",peanut,NaN
2,b9881c90-d7fa-4b77-823f-45279cbd594e,"(E,E,Z)-2,4,6-nonatrienal, 5-methyl-2-(E)-hept...","CC/C=C\C=C/C=C\C=O, CCC(C)C(=O)/C=C/C, CCC(C)C...",NaN,NaN,hazelnut;tomato;peanut;mango,smell_identification,"Given the molecules CC/C=C\C=C/C=C\C=O, CCC(C)...","Given the molecules (E,E,Z)-2,4,6-nonatrienal,...",hazelnut,NaN
3,a1ce026c-eb4d-49f5-ad0f-ed952ea353ec,"phenylacetic acid, 4-methylphenol (p-cresol), ...","C1=CC=C(C=C1)CC(=O)O, CC1=CC=C(C=C1)O, CCCC(=O...",NaN,NaN,onion;peach;cheese;tomato,smell_identification,"Given the molecules C1=CC=C(C=C1)CC(=O)O, CC1=...","Given the molecules phenylacetic acid, 4-methy...",tomato,NaN
4,aeab9337-d60c-4318-b6b4-5ae7695773e7,"methyl 2-methylbutanoate, ethyl 2-methylbutano...","CCC(C)C(=O)OC, CCC(C)C(=O)OCC, CCCC(=O)OCC, CC...",NaN,NaN,whisky;parsley;apple;melon,smell_identification,"Given the molecules CCC(C)C(=O)OC, CCC(C)C(=O)...","Given the molecules methyl 2-methylbutanoate, ...",apple,NaN


In [ ]:
import time, threading
from collections import deque
from concurrent.futures import ThreadPoolExecutor, as_completed
from openai import APIError, RateLimitError, APITimeoutError

# ============ CONFIGURATION ============
# Model options:
# - "grok-3-mini" (supports reasoning_effort: "low" or "high")
# - "grok-3" (no reasoning_effort - will error if specified)
# - "grok-4" (always max reasoning, no parameter needed)

model_name = "grok-4-1-fast-reasoning"
REASONING_EFFORT = "low"  # Only for grok-3-mini: "low" or "high"
USE_REASONING_EFFORT = False # Set to False for grok-3 or grok-4

CSV_PATH = f"{model_name.replace('-', '_')}_reasoning_{REASONING_EFFORT}_OI_Benchmark.csv"

MAX_WORKERS = 4
RPM_LIMIT = 30
SAVE_EVERY = 5

# ============ COLUMNS FOR LOGGING ============
if 'answer_to_prompt_1' not in df.columns:
    df['answer_to_prompt_1'] = None
if 'answer_to_prompt_2' not in df.columns:
    df['answer_to_prompt_2'] = None
if 'input_tokens_1' not in df.columns:
    df['input_tokens_1'] = None
if 'input_tokens_2' not in df.columns:
    df['input_tokens_2'] = None
if 'output_tokens_1' not in df.columns:
    df['output_tokens_1'] = None
if 'output_tokens_2' not in df.columns:
    df['output_tokens_2'] = None
if 'reasoning_tokens_1' not in df.columns:
    df['reasoning_tokens_1'] = None
if 'reasoning_tokens_2' not in df.columns:
    df['reasoning_tokens_2'] = None

df.reset_index(drop=True, inplace=True)

# ============ RATE LIMITER ============
class RPMLimiter:
    def __init__(self, rpm):
        self.rpm = rpm
        self.calls = deque()
        self.lock = threading.Lock()

    def wait(self):
        while True:
            with self.lock:
                now = time.time()
                while self.calls and now - self.calls[0] >= 60:
                    self.calls.popleft()
                if len(self.calls) < self.rpm:
                    self.calls.append(now)
                    return
            time.sleep(0.05)

rpm_limiter = RPMLimiter(RPM_LIMIT)
save_lock = threading.Lock()

def save():
    with save_lock:
        df.to_csv(CSV_PATH, index=False)

# ============ API CALL - RETURNS ANSWER + TOKEN USAGE ============
def ask_until_success(prompt):
    """Returns: (answer_text, input_tokens, output_tokens, reasoning_tokens)"""
    while True:
        try:
            rpm_limiter.wait()
            
            # Build request
            request_params = {
                "model": model_name,
                "max_tokens": 8192,
                "messages": [{"role": "user", "content": prompt}]
            }
            
            # Add reasoning_effort only for grok-3-mini
            if USE_REASONING_EFFORT:
                request_params["reasoning_effort"] = REASONING_EFFORT
            
            response = client.chat.completions.create(**request_params)
            
            text = response.choices[0].message.content.strip() if response.choices else ""
            
            # Extract token usage
            input_tokens = 0
            output_tokens = 0
            reasoning_tokens = 0
            
            if hasattr(response, 'usage') and response.usage:
                input_tokens = getattr(response.usage, 'prompt_tokens', 0) or 0
                output_tokens = getattr(response.usage, 'completion_tokens', 0) or 0
                
                # Get reasoning tokens from completion_tokens_details
                if hasattr(response.usage, 'completion_tokens_details'):
                    details = response.usage.completion_tokens_details
                    if details:
                        reasoning_tokens = getattr(details, 'reasoning_tokens', 0) or 0
            
            if text:
                return text, input_tokens, output_tokens, reasoning_tokens
            
            print("⛔ Empty response — retrying in 4s…")
            time.sleep(4)

        except RateLimitError as e:
            print(f"⚠️ Rate limit — retrying in 15s: {e}")
            time.sleep(15)
        except (APIError, APITimeoutError) as e:
            print(f"⚠️ API issue — retrying in 4s: {e}")
            time.sleep(4)
        except Exception as e:
            print(f"❌ Error — retrying in 4s: {e}")
            time.sleep(4)

# ============ WORKER ============
row_lock = threading.Lock()
progress = {"count": 0}

def process_row(i):
    if pd.notna(df.at[i, 'answer_to_prompt_1']) and pd.notna(df.at[i, 'answer_to_prompt_2']):
        return

    p1 = str(df.at[i, 'prompt.1'])
    p2 = str(df.at[i, 'prompt.2'])
    opt = str(df.at[i, 'OPTIONS'])

    if "OPTIONS" in p1: p1 = p1.replace("OPTIONS", opt)
    if "OPTIONS" in p2: p2 = p2.replace("OPTIONS", opt)

    a1, in1, out1, reason1 = ask_until_success(p1)
    a2, in2, out2, reason2 = ask_until_success(p2)

    with row_lock:
        df.at[i, 'answer_to_prompt_1'] = a1
        df.at[i, 'answer_to_prompt_2'] = a2
        df.at[i, 'input_tokens_1'] = in1
        df.at[i, 'input_tokens_2'] = in2
        df.at[i, 'output_tokens_1'] = out1
        df.at[i, 'output_tokens_2'] = out2
        df.at[i, 'reasoning_tokens_1'] = reason1
        df.at[i, 'reasoning_tokens_2'] = reason2
        progress["count"] += 1

        if progress["count"] % SAVE_EVERY == 0:
            save()
            print(f"💾 Saved @ {progress['count']} rows | Reasoning: {reason1}, {reason2}")

# ============ RUN ============
todo = [i for i in range(len(df))
        if pd.isna(df.at[i, 'answer_to_prompt_1']) or pd.isna(df.at[i, 'answer_to_prompt_2'])]

print(f"🚀 {len(todo)} rows to process")
print(f"🤖 Model: {model_name}")
if USE_REASONING_EFFORT:
    print(f"🧠 Reasoning effort: {REASONING_EFFORT}")
print(f"📁 Output: {CSV_PATH}")

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [executor.submit(process_row, i) for i in todo]
    for idx, _ in enumerate(as_completed(futures), 1):
        if idx % 10 == 0:
            print(f"Progress: {idx}/{len(todo)}")

save()

# Final verification
missing = df[df['answer_to_prompt_1'].isna() | df['answer_to_prompt_2'].isna()].index.tolist()
while missing:
    print(f"🔄 Fixing {len(missing)} missing...")
    for i in missing:
        process_row(i)
        save()
    missing = df[df['answer_to_prompt_1'].isna() | df['answer_to_prompt_2'].isna()].index.tolist()

save()
print(f"✅ COMPLETE — Saved to {CSV_PATH}")

🚀 1010 rows to process
🤖 Model: grok-4-1-fast-reasoning
📁 Output: grok_4_1_fast_reasoning_reasoning_low_OI_Benchmark.csv
💾 Saved @ 5 rows | Reasoning: 1712, 796
💾 Saved @ 10 rows | Reasoning: 6726, 2407
Progress: 10/1010
💾 Saved @ 15 rows | Reasoning: 2001, 787
💾 Saved @ 20 rows | Reasoning: 7646, 3155
Progress: 20/1010
💾 Saved @ 25 rows | Reasoning: 1802, 352
💾 Saved @ 30 rows | Reasoning: 3115, 3754
Progress: 30/1010
💾 Saved @ 35 rows | Reasoning: 5566, 633
💾 Saved @ 40 rows | Reasoning: 941, 504
Progress: 40/1010
💾 Saved @ 45 rows | Reasoning: 1288, 587
💾 Saved @ 50 rows | Reasoning: 1226, 474
Progress: 50/1010
💾 Saved @ 55 rows | Reasoning: 4438, 2478
💾 Saved @ 60 rows | Reasoning: 1313, 604
Progress: 60/1010
💾 Saved @ 65 rows | Reasoning: 1857, 730
💾 Saved @ 70 rows | Reasoning: 1174, 1544
Progress: 70/1010
💾 Saved @ 75 rows | Reasoning: 4686, 1646
💾 Saved @ 80 rows | Reasoning: 2520, 878
Progress: 80/1010
💾 Saved @ 85 rows | Reasoning: 1393, 594
💾 Saved @ 90 rows | Reasoning: 306

In [ ]:
# ============ SUMMARY STATISTICS ============
result = pd.read_csv(CSV_PATH)

print("\n" + "="*50)
print("📊 TOKEN USAGE SUMMARY")
print("="*50)

print(f"\nPrompt 1:")
print(f"  Mean input tokens:     {result['input_tokens_1'].mean():.1f}")
print(f"  Mean output tokens:    {result['output_tokens_1'].mean():.1f}")
print(f"  Mean reasoning tokens: {result['reasoning_tokens_1'].mean():.1f}")

print(f"\nPrompt 2:")
print(f"  Mean input tokens:     {result['input_tokens_2'].mean():.1f}")
print(f"  Mean output tokens:    {result['output_tokens_2'].mean():.1f}")
print(f"  Mean reasoning tokens: {result['reasoning_tokens_2'].mean():.1f}")

total_input = result['input_tokens_1'].sum() + result['input_tokens_2'].sum()
total_output = result['output_tokens_1'].sum() + result['output_tokens_2'].sum()
total_reasoning = result['reasoning_tokens_1'].sum() + result['reasoning_tokens_2'].sum()

print(f"\nTOTAL TOKENS:")
print(f"  Input:     {total_input:,}")
print(f"  Output:    {total_output:,}")
print(f"  Reasoning: {total_reasoning:,}")